# Tema 8 - Diseño Analítico de Controladores

**Asignatura:** Fundamentos de Control - GIERM

---

## Objetivos de aprendizaje

- Traducir especificaciones temporales (sobreoscilación, tiempo de establecimiento) a ubicación de polos en el plano $s$
- Diseñar controladores P, PI y PID para sistemas de 1er y 2o orden
- Comprender y trazar el lugar de las raíces (*root locus*) de un sistema
- Analizar la cancelación polo-cero y sus riesgos
- Evaluar el efecto de ceros adicionales en la respuesta transitoria

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Wedge
from matplotlib.lines import Line2D
from scipy import signal

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'
COLOR_RECTA = '#cb181d'
COLOR_PUNTO = '#238b45'
COLOR_ZONA = '#a6cee3'
COLOR_CERO = '#ff7f00'

print('Configuración lista.')

---

## 1. De especificaciones temporales a ubicación de polos

Las especificaciones de diseño más habituales son:

| Especificación | Símbolo | Relación con polos |
|----------------|---------|--------------------|
| Sobreoscilación | $SO\%$ | $\zeta = \dfrac{-\ln(SO/100)}{\sqrt{\pi^2 + \ln^2(SO/100)}}$ |
| Tiempo de establecimiento (2%) | $t_s$ | $\zeta\omega_n = \dfrac{4}{t_s}$ |
| Tiempo de pico | $t_p$ | $\omega_d = \dfrac{\pi}{t_p}$ |
| Frecuencia natural | $\omega_n$ | $\omega_n = \dfrac{\zeta\omega_n}{\zeta}$ |

Los polos dominantes deseados en el plano $s$ son:

$$\boxed{s_{1,2} = -\zeta\omega_n \pm j\omega_n\sqrt{1-\zeta^2} = -\sigma \pm j\omega_d}$$

donde $\sigma = \zeta\omega_n$ y $\omega_d = \omega_n\sqrt{1-\zeta^2}$.

In [ ]:
# Plano s: regiones de especificaciones
fig, ax = plt.subplots(figsize=(10, 8))

# Especificaciones de ejemplo: SO% <= 10%, ts <= 2s
SO = 10  # %
ts = 2   # s

zeta_min = -np.log(SO / 100) / np.sqrt(np.pi**2 + np.log(SO / 100)**2)
sigma_min = 4 / ts  # zeta*wn mínimo

# Dibujar ejes imaginarios
ax.axhline(0, color='k', lw=0.8)
ax.axvline(0, color='k', lw=0.8)

# Región de sigma: Re(s) <= -sigma_min
ax.axvline(-sigma_min, color=COLOR_RECTA, ls='--', lw=2, label=rf'$\sigma \geq {sigma_min:.1f}$ ($t_s \leq {ts}$ s)')
ax.fill_betweenx([-8, 8], -10, -sigma_min, alpha=0.08, color=COLOR_RECTA)

# Región de zeta: líneas desde el origen con ángulo arccos(zeta)
theta = np.arccos(zeta_min)
r_line = 8
ax.plot([0, -r_line * np.cos(theta)], [0, r_line * np.sin(theta)], '--', color=COLOR_PRINCIPAL, lw=2)
ax.plot([0, -r_line * np.cos(theta)], [0, -r_line * np.sin(theta)], '--', color=COLOR_PRINCIPAL, lw=2,
        label=rf'$\zeta \geq {zeta_min:.3f}$ ($SO \leq {SO}\%$)')

# Sombrear región válida (sector entre las líneas de zeta, a la izquierda de sigma_min)
theta_fill = np.linspace(-theta, theta, 200)
r_fill = 8
x_fill = np.concatenate([[-sigma_min, -sigma_min], -r_fill * np.cos(theta_fill[::-1])])
y_fill = np.concatenate([[-r_fill * np.sin(theta), r_fill * np.sin(theta)], r_fill * np.sin(theta_fill[::-1])])

# Región válida simplificada
angles_upper = np.linspace(0, theta, 100)
angles_lower = np.linspace(0, -theta, 100)
sector_x = np.concatenate([[-sigma_min], -r_fill * np.cos(angles_upper), [-r_fill * np.cos(theta), -sigma_min, -sigma_min],
                            -r_fill * np.cos(angles_lower[::-1]), [-r_fill * np.cos(theta)]])
sector_y = np.concatenate([[0], r_fill * np.sin(angles_upper), [r_fill * np.sin(theta), r_fill * np.sin(theta), 0],
                            r_fill * np.sin(angles_lower[::-1]), [-r_fill * np.sin(theta)]])

# Zona válida
wedge_upper = Wedge((0, 0), r_fill, 180 - np.degrees(theta), 180, alpha=0.12, color=COLOR_PUNTO)
wedge_lower = Wedge((0, 0), r_fill, 180, 180 + np.degrees(theta), alpha=0.12, color=COLOR_PUNTO)
ax.add_patch(wedge_upper)
ax.add_patch(wedge_lower)

# Polos de ejemplo en la zona válida
wn = sigma_min / zeta_min
wd = wn * np.sqrt(1 - zeta_min**2)
ax.plot(-sigma_min, wd, 'x', color=COLOR_PUNTO, ms=14, mew=3, zorder=5)
ax.plot(-sigma_min, -wd, 'x', color=COLOR_PUNTO, ms=14, mew=3, zorder=5)
ax.annotate(f'Polo deseado\n$s = -{sigma_min:.1f} + j{wd:.2f}$',
            xy=(-sigma_min, wd), xytext=(-sigma_min - 2.5, wd + 1.5),
            fontsize=11, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2),
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))

ax.set_xlabel(r'$\mathrm{Re}(s)$')
ax.set_ylabel(r'$\mathrm{Im}(s)$')
ax.set_title(r'Plano $s$: Regiones de especificaciones ($SO \leq 10\%$, $t_s \leq 2$ s)')
ax.set_xlim(-8, 2)
ax.set_ylim(-6, 6)
ax.legend(fontsize=11, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

**Interpretación de la gráfica:**

- La **línea vertical roja** marca $\mathrm{Re}(s) = -\sigma_{\min} = -2$. Los polos deben estar a la **izquierda** para cumplir $t_s \leq 2$ s.
- Las **líneas diagonales azules** forman un sector angular. Los polos deben estar **dentro** del sector para cumplir $SO \leq 10\%$.
- La **zona verde sombreada** es la intersección: cualquier polo en esa región cumple ambas especificaciones.

**Resumen de conversión:**

$$\boxed{SO\% \to \zeta \to \theta = \arccos(\zeta)} \qquad \boxed{t_s \to \sigma_{\min} = \frac{4}{t_s}}$$

---

## 2. Control de sistemas de 1er orden

Planta genérica de primer orden:

$$G(s) = \frac{K}{\tau s + 1}$$

### 2.1 Control proporcional (P)

Controlador: $C(s) = K_p$

Función de transferencia en bucle cerrado:

$$\boxed{G_{BC}(s) = \frac{K K_p}{\tau s + 1 + K K_p} = \frac{\dfrac{K K_p}{1 + K K_p}}{\dfrac{\tau}{1 + K K_p} s + 1}}$$

**Efecto del control P:**
- La **constante de tiempo** se reduce: $\tau_{BC} = \dfrac{\tau}{1 + K K_p}$ $\to$ el sistema es más rápido
- La **ganancia DC** no es 1: $G_{BC}(0) = \dfrac{K K_p}{1 + K K_p} < 1$
- **Error en régimen permanente** ante escalón unitario:

$$\boxed{e_{ss} = \frac{1}{1 + K K_p}}$$

**Conclusión:** el control P **no puede eliminar** el error estacionario en un sistema de tipo 0.

In [ ]:
# Comparación antes/después con control P
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

K_planta = 2.0
tau = 1.0
Kp_vals = [1, 3, 10]

t = np.linspace(0, 6, 500)

# Respuesta en lazo abierto
y_ol = K_planta * (1 - np.exp(-t / tau))
axes[0].plot(t, y_ol, color='gray', lw=2.5, ls='--', label=r'Lazo abierto $G(s)$')

colores_kp = ['#6baed6', '#3182bd', '#08519c']
for Kp, col in zip(Kp_vals, colores_kp):
    tau_cl = tau / (1 + K_planta * Kp)
    gain_cl = K_planta * Kp / (1 + K_planta * Kp)
    y_cl = gain_cl * (1 - np.exp(-t / tau_cl))
    axes[0].plot(t, y_cl, color=col, lw=2, label=rf'$K_p = {Kp}$, $e_{{ss}} = {1/(1+K_planta*Kp):.3f}$')

axes[0].axhline(1, color=COLOR_RECTA, ls=':', lw=1.5, label='Referencia')
axes[0].set_xlabel(r'Tiempo (s)')
axes[0].set_ylabel(r'Salida $y(t)$')
axes[0].set_title(r'Respuesta al escalón con control P: $G(s) = 2/(s+1)$')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.15)

# Mapa de polos antes/después
axes[1].axhline(0, color='k', lw=0.8)
axes[1].axvline(0, color='k', lw=0.8)
polo_ol = -1 / tau
axes[1].plot(polo_ol, 0, 'x', color='gray', ms=14, mew=3, label='Polo lazo abierto')

for Kp, col in zip(Kp_vals, colores_kp):
    polo_cl = -(1 + K_planta * Kp) / tau
    axes[1].plot(polo_cl, 0, 'x', color=col, ms=14, mew=3, label=rf'Polo con $K_p = {Kp}$: $s = {polo_cl:.1f}$')

axes[1].set_xlabel(r'$\mathrm{Re}(s)$')
axes[1].set_ylabel(r'$\mathrm{Im}(s)$')
axes[1].set_title('Mapa de polos: efecto del control P')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-1, 1)

plt.tight_layout()
plt.show()

### 2.2 Control PI (Proporcional-Integral)

Controlador:

$$C(s) = K_p\left(1 + \frac{1}{T_I s}\right) = K_p \cdot \frac{T_I s + 1}{T_I s}$$

El PI introduce:
- Un **polo en el origen** ($s = 0$): convierte el sistema en tipo 1 $\to$ **error cero** ante escalón
- Un **cero** en $s = -1/T_I$

**Estrategia de diseño:** si elegimos $T_I = \tau$ (la constante de tiempo de la planta), el cero del controlador **cancela** el polo de la planta:

$$C(s) \cdot G(s) = K_p \cdot \frac{\cancel{\tau s + 1}}{T_I s} \cdot \frac{K}{\cancel{\tau s + 1}} = \frac{K K_p}{T_I s}$$

Bucle cerrado:

$$\boxed{G_{BC}(s) = \frac{K K_p / T_I}{s + K K_p / T_I} = \frac{1}{\dfrac{T_I}{K K_p} s + 1}}$$

**Resultado:** sistema de 1er orden con ganancia DC = 1 (error cero) y constante de tiempo $\tau_{BC} = T_I / (K K_p)$.

In [ ]:
# Comparación P vs PI
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

K_planta = 2.0
tau = 1.0
Kp = 3.0
TI = tau  # Cancelación polo-cero

t = np.linspace(0, 6, 500)

# Control P
gain_p = K_planta * Kp / (1 + K_planta * Kp)
tau_p = tau / (1 + K_planta * Kp)
y_P = gain_p * (1 - np.exp(-t / tau_p))

# Control PI (con cancelación: 1er orden, ganancia 1)
tau_pi = TI / (K_planta * Kp)
y_PI = 1 - np.exp(-t / tau_pi)

axes[0].plot(t, y_P, color=COLOR_PRINCIPAL, lw=2.5, label=rf'Control P ($K_p = {Kp}$)')
axes[0].plot(t, y_PI, color=COLOR_PUNTO, lw=2.5, label=rf'Control PI ($K_p = {Kp}$, $T_I = {TI}$)')
axes[0].axhline(1, color=COLOR_RECTA, ls=':', lw=1.5, label='Referencia')
axes[0].annotate(f'Error P = {1-gain_p:.3f}', xy=(5, gain_p), xytext=(3.5, gain_p - 0.12),
                 fontsize=11, color=COLOR_PRINCIPAL, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=COLOR_PRINCIPAL, lw=1.5))
axes[0].annotate('Error PI = 0', xy=(5, 1), xytext=(3.5, 0.85),
                 fontsize=11, color=COLOR_PUNTO, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=1.5))
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel(r'Salida $y(t)$')
axes[0].set_title(r'Respuesta al escalón: P vs PI')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.15)

# Mapa de polos
axes[1].axhline(0, color='k', lw=0.8)
axes[1].axvline(0, color='k', lw=0.8)
axes[1].plot(-1, 0, 'x', color='gray', ms=14, mew=3, label='Polo planta (LA)')
polo_pi = -K_planta * Kp / TI
axes[1].plot(polo_pi, 0, 'x', color=COLOR_PUNTO, ms=14, mew=3, label=rf'Polo BC (PI): $s = {polo_pi:.1f}$')
# Cero del PI que cancela
axes[1].plot(-1/TI, 0, 'o', color=COLOR_CERO, ms=12, mew=2, fillstyle='none',
             label=rf'Cero PI: $s = {-1/TI:.1f}$ (cancela polo)')

axes[1].set_xlabel(r'$\mathrm{Re}(s)$')
axes[1].set_ylabel(r'$\mathrm{Im}(s)$')
axes[1].set_title('Mapa de polos y ceros con PI')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-1, 1)

plt.tight_layout()
plt.show()

---

### 2.3 Ejemplo resuelto: Diseño PI para sistema de 1er orden

#### Ejercicio resuelto: PI para $G(s) = 2/(s+1)$

**Datos:** $G(s) = \dfrac{2}{s + 1}$. Diseñar un PI tal que $e_{ss} = 0$ y $t_s \leq 2$ s (criterio del 2%).

**Paso 1:** Identificar la planta. $K = 2$, $\tau = 1$ s, polo en $s = -1$.

**Paso 2:** Elegir $T_I = \tau = 1$ s para cancelar el polo de la planta.

$$C(s) = K_p \cdot \frac{s + 1}{s}$$

**Paso 3:** Con la cancelación, el bucle cerrado queda:

$$G_{BC}(s) = \frac{2 K_p}{s + 2 K_p}$$

El polo de BC está en $s = -2K_p$.

**Paso 4:** Imponer $t_s \leq 2$ s:

$$t_s = \frac{4}{2K_p} \leq 2 \implies K_p \geq 1$$

**Paso 5:** Elegimos $K_p = 2$ (margen de seguridad):

$$\boxed{C(s) = 2 \cdot \frac{s + 1}{s} = \frac{2s + 2}{s}}$$

**Verificación:**
- $G_{BC}(s) = \dfrac{4}{s + 4}$ $\to$ polo en $s = -4$ $\to$ $t_s = \dfrac{4}{4} = 1$ s $\leq 2$ s $\checkmark$
- $G_{BC}(0) = \dfrac{4}{4} = 1$ $\to$ $e_{ss} = 0$ $\checkmark$

In [ ]:
# Verificación gráfica del diseño PI
fig, ax = plt.subplots(figsize=(12, 6))

t = np.linspace(0, 5, 500)

# Lazo abierto
y_ol = 2 * (1 - np.exp(-t))
ax.plot(t, y_ol, color='gray', lw=2, ls='--', label='Lazo abierto')

# Con PI diseñado: Kp=2, TI=1 -> polo en s=-4
y_pi = 1 - np.exp(-4 * t)
ax.plot(t, y_pi, color=COLOR_PUNTO, lw=2.5, label=r'PI: $C(s) = 2(s+1)/s$')

# Marcar ts
ts_real = 4 / 4
ax.axvline(ts_real, color=COLOR_RECTA, ls='--', lw=1.5, alpha=0.7, label=rf'$t_s = {ts_real:.1f}$ s')
ax.axhline(1, color=COLOR_RECTA, ls=':', lw=1.5, alpha=0.5)
ax.axhspan(0.98, 1.02, alpha=0.1, color=COLOR_PUNTO, label=r'Banda $\pm 2\%$')

ax.set_xlabel('Tiempo (s)')
ax.set_ylabel(r'Salida $y(t)$')
ax.set_title(r'Diseño PI verificado: $G(s) = 2/(s+1)$, $e_{ss} = 0$, $t_s = 1$ s')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.2)
plt.tight_layout()
plt.show()

---

## 3. Control de sistemas de 2o orden

### 3.1 Método de asignación de polos (*pole placement*)

Para una planta de 2o orden:

$$G(s) = \frac{K}{s^2 + a_1 s + a_0}$$

Se desea que el bucle cerrado tenga polos en $s_{1,2} = -\sigma \pm j\omega_d$, lo que corresponde al polinomio característico deseado:

$$\boxed{D(s) = s^2 + 2\zeta\omega_n s + \omega_n^2}$$

**Procedimiento:**

1. Calcular $\zeta$ y $\omega_n$ a partir de las especificaciones
2. Escribir el polinomio característico deseado: $D(s) = s^2 + 2\zeta\omega_n s + \omega_n^2$
3. Escribir el polinomio característico del sistema realimentado con el controlador
4. Igualar coeficientes y resolver para los parámetros del controlador

### 3.2 Ejemplo resuelto: Asignación de polos en sistema de 2o orden

#### Ejercicio resuelto: PD para planta $G(s) = 1/(s^2 + 3s + 2)$

**Datos:** $G(s) = \dfrac{1}{s^2 + 3s + 2} = \dfrac{1}{(s+1)(s+2)}$. Diseñar un controlador PD $C(s) = K_p + K_d s$ para que $SO \leq 5\%$ y $t_s \leq 1$ s.

**Paso 1:** Obtener $\zeta$ y $\sigma$:

$$\zeta = \frac{-\ln(0.05)}{\sqrt{\pi^2 + \ln^2(0.05)}} = \frac{2.996}{\sqrt{9.870 + 8.976}} = 0.690$$

$$\sigma = \frac{4}{t_s} = \frac{4}{1} = 4$$

**Paso 2:** Calcular $\omega_n$ y $\omega_d$:

$$\omega_n = \frac{\sigma}{\zeta} = \frac{4}{0.690} = 5.797 \;\text{rad/s}$$

$$\omega_d = \omega_n\sqrt{1 - \zeta^2} = 5.797 \times 0.724 = 4.197 \;\text{rad/s}$$

**Paso 3:** Polinomio característico deseado:

$$D(s) = s^2 + 2(0.690)(5.797)s + 5.797^2 = s^2 + 8.0 s + 33.60$$

**Paso 4:** Polinomio del bucle cerrado con PD:

$$1 + C(s)G(s) = 1 + \frac{(K_p + K_d s)}{s^2 + 3s + 2} = \frac{s^2 + (3 + K_d)s + (2 + K_p)}{s^2 + 3s + 2} = 0$$

Polinomio característico: $s^2 + (3 + K_d)s + (2 + K_p)$

**Paso 5:** Igualar coeficientes:

$$3 + K_d = 8.0 \implies \boxed{K_d = 5.0}$$

$$2 + K_p = 33.60 \implies \boxed{K_p = 31.60}$$

**Controlador:** $C(s) = 31.60 + 5.0 s$

In [ ]:
# Verificación: asignación de polos en 2o orden
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Parámetros
Kp_pd = 31.60
Kd_pd = 5.0

# Sistema original en lazo abierto
num_ol = [1]
den_ol = [1, 3, 2]
sys_ol = signal.TransferFunction(num_ol, den_ol)

# Sistema en bucle cerrado con PD
num_cl = [Kd_pd, Kp_pd]
den_cl = [1, 3 + Kd_pd, 2 + Kp_pd]
sys_cl = signal.TransferFunction(num_cl, den_cl)

t = np.linspace(0, 3, 500)
t_ol, y_ol = signal.step(sys_ol, T=t)
t_cl, y_cl = signal.step(sys_cl, T=t)

axes[0].plot(t_ol, y_ol, color='gray', lw=2, ls='--', label='Sin controlador')
axes[0].plot(t_cl, y_cl, color=COLOR_PUNTO, lw=2.5, label=r'Con PD ($K_p=31.6$, $K_d=5.0$)')
axes[0].axhline(1, color=COLOR_RECTA, ls=':', lw=1.5, alpha=0.5)
axes[0].axhspan(0.98, 1.02, alpha=0.1, color=COLOR_PUNTO)
axes[0].axvline(1.0, color=COLOR_RECTA, ls='--', lw=1.5, alpha=0.5, label=r'$t_s$ deseado = 1 s')
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel(r'Salida $y(t)$')
axes[0].set_title('Respuesta al escalón: antes y después del PD')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.4)

# Mapa de polos
axes[1].axhline(0, color='k', lw=0.8)
axes[1].axvline(0, color='k', lw=0.8)

# Polos originales
polos_ol = np.roots(den_ol)
axes[1].plot(polos_ol.real, polos_ol.imag, 'x', color='gray', ms=14, mew=3, label='Polos lazo abierto')

# Polos deseados / BC
polos_cl = np.roots(den_cl)
axes[1].plot(polos_cl.real, polos_cl.imag, 'x', color=COLOR_PUNTO, ms=14, mew=3, label='Polos deseados (BC)')

# Etiquetas
for p in polos_ol:
    axes[1].annotate(f's = {p.real:.1f}', xy=(p.real, p.imag), xytext=(p.real + 0.3, p.imag + 0.8),
                     fontsize=10, color='gray')
for p in polos_cl:
    axes[1].annotate(f's = {p.real:.1f}{p.imag:+.1f}j', xy=(p.real, p.imag),
                     xytext=(p.real + 0.5, p.imag + 0.8),
                     fontsize=10, color=COLOR_PUNTO, fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=1.5))

axes[1].set_xlabel(r'$\mathrm{Re}(s)$')
axes[1].set_ylabel(r'$\mathrm{Im}(s)$')
axes[1].set_title('Mapa de polos: original vs diseño PD')
axes[1].legend(fontsize=10, loc='upper left')
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

---

## 4. Lugar de las raíces (*Root Locus*)

### 4.1 Concepto

El **lugar de las raíces** muestra cómo se mueven los polos del bucle cerrado cuando la ganancia $K$ varía de 0 a $\infty$.

Para un sistema con función de transferencia en lazo abierto $KG(s)$, la ecuación característica es:

$$\boxed{1 + K G(s) = 0 \implies G(s) = -\frac{1}{K}}$$

**Propiedades principales:**
- Las ramas **comienzan** en los polos de $G(s)$ (cuando $K = 0$)
- Las ramas **terminan** en los ceros de $G(s)$ o en el infinito (cuando $K \to \infty$)
- Número de ramas = número de polos de $G(s)$
- El lugar es **simétrico** respecto al eje real
- Un punto del eje real pertenece al lugar si tiene un número **impar** de polos + ceros a su derecha

### 4.2 Cálculo numérico

Para trazar el lugar de las raíces numéricamente, resolvemos $1 + K G(s) = 0$ para muchos valores de $K$:

Si $G(s) = \dfrac{N(s)}{D(s)}$, entonces $D(s) + K \cdot N(s) = 0$ define el polinomio característico para cada $K$.

In [ ]:
# Lugar de las raíces: G(s) = 1/[s(s+1)(s+2)]
fig, ax = plt.subplots(figsize=(10, 8))

# Polinomios de G(s) = 1/[s(s+1)(s+2)] = 1/(s^3 + 3s^2 + 2s)
num_coeffs = [1]       # numerador: 1
den_coeffs = [1, 3, 2, 0]  # denominador: s^3 + 3s^2 + 2s

# Barrido de K
K_values = np.concatenate([
    np.linspace(0.001, 1, 500),
    np.linspace(1, 10, 500),
    np.linspace(10, 100, 500)
])

# Para cada K, resolver D(s) + K*N(s) = 0
all_roots = []
for K in K_values:
    # D(s) + K*N(s): sumar coeficientes
    # den tiene grado 3, num tiene grado 0 -> pad num
    num_padded = np.zeros(len(den_coeffs))
    num_padded[-len(num_coeffs):] = num_coeffs
    char_poly = np.array(den_coeffs) + K * num_padded
    roots = np.roots(char_poly)
    all_roots.append(roots)

all_roots = np.array(all_roots)

# Dibujar cada rama
colores_ramas = [COLOR_PRINCIPAL, COLOR_RECTA, COLOR_PUNTO]
for i in range(all_roots.shape[1]):
    ax.plot(all_roots[:, i].real, all_roots[:, i].imag, '.', color=colores_ramas[i % 3],
            ms=1.5, alpha=0.6)

# Marcar polos de lazo abierto (inicio)
polos_la = np.roots(den_coeffs)
ax.plot(polos_la.real, polos_la.imag, 'x', color='k', ms=14, mew=3, zorder=5, label='Polos LA (inicio)')

# Punto de separación (breakaway): entre s=-1 y s=0, y entre s=-2 y s=-1
# dK/ds = 0 donde K = -D(s)/N(s) = -(s^3+3s^2+2s)
# dK/ds = -(3s^2 + 6s + 2) = 0 -> s = (-6 ± sqrt(36-24))/6 = (-6 ± 3.46)/6
s_break1 = (-6 + np.sqrt(12)) / 6  # -0.423
s_break2 = (-6 - np.sqrt(12)) / 6  # -1.577
ax.plot(s_break1, 0, 'd', color=COLOR_CERO, ms=10, zorder=5, label=f'Punto separación: $s = {s_break1:.3f}$')
ax.plot(s_break2, 0, 'd', color=COLOR_CERO, ms=10, zorder=5, label=f'Punto separación: $s = {s_break2:.3f}$')

# Asíntotas: n-m = 3-0 = 3 asíntotas
# Centro: (sum polos - sum ceros)/(n-m) = (-3)/3 = -1
# Ángulos: (2k+1)*180/3 = 60, 180, 300 grados
ax.axvline(-1, color='gray', ls=':', lw=1, alpha=0.5, label=r'Centro asíntotas: $\sigma_a = -1$')

# Marcar K crítico (cruce con eje imaginario): Routh -> K_crit = 6
K_crit = 6.0
num_pad = np.zeros(len(den_coeffs))
num_pad[-len(num_coeffs):] = num_coeffs
roots_crit = np.roots(np.array(den_coeffs) + K_crit * num_pad)
for r in roots_crit:
    if abs(r.real) < 0.1:
        ax.plot(r.real, r.imag, 's', color=COLOR_RECTA, ms=10, zorder=5)
        ax.annotate(f'$K_{{crit}} = {K_crit:.0f}$\n$s = j{abs(r.imag):.2f}$',
                    xy=(r.real, r.imag), xytext=(1.5, r.imag),
                    fontsize=10, color=COLOR_RECTA, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=COLOR_RECTA, lw=1.5),
                    bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_RECTA))

ax.set_xlabel(r'$\mathrm{Re}(s)$')
ax.set_ylabel(r'$\mathrm{Im}(s)$')
ax.set_title(r'Lugar de las raíces: $G(s) = 1/[s(s+1)(s+2)]$')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xlim(-4, 2)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

**Análisis del lugar de las raíces:**

- **3 ramas** parten de los polos $s = 0$, $s = -1$, $s = -2$
- **Puntos de separación** (*breakaway*) en $s = -0.423$ y $s = -1.577$
- **Centro de asíntotas:** $\sigma_a = \dfrac{0 + (-1) + (-2)}{3} = -1$
- **Ángulos de asíntotas:** $60°$, $180°$, $300°$
- **$K$ crítico** (cruce con eje imaginario): $K_{crit} = 6$ $\to$ para $K > 6$ el sistema es **inestable**

**Error frecuente:** olvidar que el lugar es solo para $K > 0$. Para $K < 0$ el lugar es diferente.

In [ ]:
# Efecto de K sobre la respuesta: antes y después de K_crit
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

t = np.linspace(0, 15, 1000)
K_vals_demo = [1, 3, 5, 6, 8]
colores_k = ['#6baed6', '#3182bd', '#08519c', COLOR_RECTA, '#d94801']

for K_val, col in zip(K_vals_demo, colores_k):
    # BC: G_cl = K / (s^3 + 3s^2 + 2s + K)
    num_cl = [K_val]
    den_cl = [1, 3, 2, K_val]
    sys_cl = signal.TransferFunction(num_cl, den_cl)
    t_out, y_out = signal.step(sys_cl, T=t)
    ls = '--' if K_val >= 6 else '-'
    axes[0].plot(t_out, y_out, color=col, lw=2, ls=ls, label=rf'$K = {K_val}$')

axes[0].axhline(1, color='gray', ls=':', lw=1)
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel(r'Salida $y(t)$')
axes[0].set_title(r'Respuesta al escalón para distintos $K$')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-2, 3)

# Mapa de polos para cada K
axes[1].axhline(0, color='k', lw=0.8)
axes[1].axvline(0, color='k', lw=0.8)
for K_val, col in zip(K_vals_demo, colores_k):
    polos = np.roots([1, 3, 2, K_val])
    axes[1].plot(polos.real, polos.imag, 'x', color=col, ms=12, mew=3, label=rf'$K = {K_val}$')

axes[1].set_xlabel(r'$\mathrm{Re}(s)$')
axes[1].set_ylabel(r'$\mathrm{Im}(s)$')
axes[1].set_title(r'Polos del BC para distintos $K$')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

---

## 5. Cancelación polo-cero

### 5.1 Cuándo es válida

Si el controlador introduce un cero que cancela un polo de la planta:

- **Polo estable** ($\mathrm{Re}(s) < 0$): la cancelación es **aceptable** en la práctica. El modo cancelado decae exponencialmente.
- **Polo inestable** ($\mathrm{Re}(s) > 0$): la cancelación es **peligrosa**. Cualquier perturbación excita el modo oculto, que crece sin control.

### 5.2 Por qué es peligrosa con polos inestables

Aunque la FT en bucle cerrado no muestre el polo inestable, este sigue presente en la **realización interna** del sistema. Una perturbación infinitesimal lo excita y la salida diverge.

$$\boxed{\text{NUNCA cancelar polos inestables con ceros del controlador}}$$

In [ ]:
# Cancelación polo-cero: polo estable vs inestable
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

t = np.linspace(0, 10, 1000)

# Caso 1: Cancelación de polo estable s = -1
# Planta: G = 1/[(s+1)(s+3)], Controlador: C = 5(s+1)/s
# Ideal: BC = 5/[s(s+3) + 5] = 5/(s^2 + 3s + 5)
# Con perturbación: modo e^{-t} presente pero decae
num_ideal = [5]
den_ideal = [1, 3, 5]
sys_ideal = signal.TransferFunction(num_ideal, den_ideal)
t1, y_ideal = signal.step(sys_ideal, T=t)

# Añadir modo "oculto" con amplitud pequeña
y_real_stable = y_ideal + 0.15 * np.exp(-t)  # modo estable oculto

axes[0].plot(t1, y_ideal, color=COLOR_PRINCIPAL, lw=2.5, label='FT ideal (cancelación perfecta)')
axes[0].plot(t1, y_real_stable, color=COLOR_PUNTO, lw=2, ls='--', label='Real (con modo oculto)')
axes[0].axhline(1, color='gray', ls=':', lw=1)
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel(r'Salida $y(t)$')
axes[0].set_title(r'Cancelación de polo ESTABLE: $s = -1$ (seguro)')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.6)

# Caso 2: Cancelación de polo INESTABLE s = +1
# Planta: G = 1/[(s-1)(s+3)], Controlador intenta cancelar: C = K(s-1)/s
# FT ideal parece estable, pero el modo e^{+t} crece
num_ideal2 = [5]
den_ideal2 = [1, 3, 5]
sys_ideal2 = signal.TransferFunction(num_ideal2, den_ideal2)
t2, y_ideal2 = signal.step(sys_ideal2, T=t)

# Modo oculto INESTABLE
epsilon = 0.01  # perturbación pequeña
y_real_unstable = y_ideal2 + epsilon * np.exp(t)  # crece exponencialmente

axes[1].plot(t2, y_ideal2, color=COLOR_PRINCIPAL, lw=2.5, label='FT ideal (cancelación perfecta)')
axes[1].plot(t2, y_real_unstable, color=COLOR_RECTA, lw=2, ls='--', label=r'Real (modo oculto $e^{+t}$)')
axes[1].axhline(1, color='gray', ls=':', lw=1)
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel(r'Salida $y(t)$')
axes[1].set_title(r'Cancelación de polo INESTABLE: $s = +1$ (PELIGROSO)')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-2, 15)

plt.tight_layout()
plt.show()

---

## 6. Efecto de ceros en la respuesta del bucle cerrado

Un cero en la FT de bucle cerrado $G_{BC}(s) = \dfrac{(s - z) \cdots}{(s - p_1)(s - p_2) \cdots}$ modifica la respuesta temporal:

| Tipo de cero | Efecto |
|-------------|--------|
| Cero negativo cercano al eje imaginario | **Aumenta** la sobreoscilación y acelera la respuesta |
| Cero negativo lejano | Efecto despreciable |
| Cero positivo (no mínima fase) | Produce **undershoot** inicial (la respuesta va primero al revés) |

**Regla práctica:** un cero en $s = -z$ afecta significativamente si $|z|$ es del mismo orden que $|\sigma|$ de los polos dominantes.

In [ ]:
# Efecto de ceros sobre la respuesta
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

t = np.linspace(0, 8, 800)
wn = 2.0
zeta = 0.5

# Sistema base sin cero
num_base = [wn**2]
den_base = [1, 2*zeta*wn, wn**2]
sys_base = signal.TransferFunction(num_base, den_base)
t0, y_base = signal.step(sys_base, T=t)

# Efecto de cero en semiplano izquierdo (distintas distancias)
axes[0].plot(t0, y_base, color='gray', lw=2, ls='--', label='Sin cero')

zeros_lhp = [1, 3, 10]
colores_z = ['#e6550d', '#fd8d3c', '#fdbe85']
for z, col in zip(zeros_lhp, colores_z):
    # G(s) = wn^2 * (s+z) / [z * (s^2 + 2*zeta*wn*s + wn^2)]  (ganancia DC = 1)
    num_z = [wn**2 / z, wn**2]
    den_z = den_base
    sys_z = signal.TransferFunction(num_z, den_z)
    tz, yz = signal.step(sys_z, T=t)
    axes[0].plot(tz, yz, color=col, lw=2, label=rf'Cero en $s = -{z}$')

axes[0].axhline(1, color='gray', ls=':', lw=1)
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel(r'Salida $y(t)$')
axes[0].set_title('Efecto de ceros en semiplano izquierdo')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.8)

# Efecto de cero en semiplano derecho (no mínima fase)
axes[1].plot(t0, y_base, color='gray', lw=2, ls='--', label='Sin cero')

zeros_rhp = [0.5, 1, 3]
colores_rhp = [COLOR_RECTA, '#d94801', '#fdae6b']
for z, col in zip(zeros_rhp, colores_rhp):
    # Cero en s = +z -> (s - z)
    # G(s) = wn^2 * (-s + z) / [z * den]  (ganancia DC = 1, cero positivo)
    num_z = [-wn**2 / z, wn**2]
    den_z = den_base
    sys_z = signal.TransferFunction(num_z, den_z)
    tz, yz = signal.step(sys_z, T=t)
    axes[1].plot(tz, yz, color=col, lw=2, label=rf'Cero en $s = +{z}$ (NMP)')

axes[1].axhline(1, color='gray', ls=':', lw=1)
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel(r'Salida $y(t)$')
axes[1].set_title('Efecto de ceros en semiplano derecho (no mínima fase)')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.8, 1.8)

plt.tight_layout()
plt.show()

**Observaciones clave:**

- **Cero cercano en SPIzq** ($s = -1$): aumenta la sobreoscilación drásticamente (de ~16% a ~50%)
- **Cero lejano en SPIzq** ($s = -10$): casi no afecta la respuesta
- **Cero en SPDer** ($s = +z$): produce *undershoot* (la salida baja antes de subir). Cuanto menor $z$, mayor el *undershoot*.

**Truco para el examen:** si tras diseñar un PI aparece un cero cercano a los polos dominantes, la sobreoscilación será **mayor** que la predicha con la fórmula de 2o orden puro.

---

## 7. Ejercicios resueltos

### 7.1 Ejercicio 1: Diseño PI para sistema de 1er orden

#### Ejercicio resuelto: PI para $G(s) = 5/(2s + 1)$

**Datos:** $G(s) = \dfrac{5}{2s + 1}$. Diseñar un PI para $e_{ss} = 0$ y $t_s \leq 1$ s.

**Paso 1:** Identificar planta: $K = 5$, $\tau = 2$ s, polo en $s = -0.5$.

**Paso 2:** Elegir $T_I = \tau = 2$ s $\to$ cancelación polo-cero.

$$C(s) = K_p \cdot \frac{2s + 1}{2s} = K_p \cdot \frac{s + 0.5}{s}$$

**Paso 3:** Tras cancelación:

$$G_{BC}(s) = \frac{5K_p / 2}{s + 5K_p / 2} = \frac{1}{\dfrac{2}{5K_p}s + 1}$$

Polo BC en $s = -5K_p / 2$.

**Paso 4:** Imponer $t_s \leq 1$ s:

$$t_s = \frac{4}{5K_p / 2} = \frac{8}{5K_p} \leq 1 \implies K_p \geq 1.6$$

**Paso 5:** Elegimos $K_p = 2$:

$$\boxed{C(s) = 2 \cdot \frac{s + 0.5}{s} = \frac{2s + 1}{s}}$$

**Verificación:** $G_{BC}(s) = \dfrac{5}{s + 5}$ $\to$ $t_s = \dfrac{4}{5} = 0.8$ s $\leq 1$ s $\checkmark$, $e_{ss} = 0$ $\checkmark$.

---

### 7.2 Ejercicio 2: Diseño PD para sistema de 2o orden

#### Ejercicio resuelto: PD para $G(s) = 4/(s^2 + 2s + 5)$

**Datos:** $G(s) = \dfrac{4}{s^2 + 2s + 5}$. Diseñar PD $C(s) = K_p + K_d s$ para $SO \leq 2\%$ y $t_s \leq 2$ s.

**Paso 1:** Especificaciones $\to$ parámetros:

$$\zeta = \frac{-\ln(0.02)}{\sqrt{\pi^2 + \ln^2(0.02)}} = \frac{3.912}{\sqrt{9.870 + 15.30}} = 0.780$$

$$\sigma = \frac{4}{t_s} = \frac{4}{2} = 2 \implies \omega_n = \frac{2}{0.780} = 2.564$$

**Paso 2:** Polinomio deseado:

$$D(s) = s^2 + 2(0.780)(2.564)s + 2.564^2 = s^2 + 4.0 s + 6.576$$

**Paso 3:** Ecuación característica con PD:

$$s^2 + (2 + 4K_d)s + (5 + 4K_p) = 0$$

**Paso 4:** Igualar:

$$2 + 4K_d = 4.0 \implies \boxed{K_d = 0.5}$$

$$5 + 4K_p = 6.576 \implies \boxed{K_p = 0.394}$$

**Nota:** el cero del PD en $s = -K_p/K_d = -0.788$ está cerca de los polos dominantes, lo que aumentará algo la sobreoscilación real respecto al valor predicho.

In [ ]:
# Verificación del ejercicio 2: PD para planta de 2o orden
fig, ax = plt.subplots(figsize=(12, 6))

t = np.linspace(0, 6, 600)

# Sin controlador (BC con ganancia unitaria)
num_orig = [4]
den_orig = [1, 2, 5 + 4]  # 1 + G(s) -> s^2 + 2s + 9 (con K=1 proporcional)
# Mejor: mostramos lazo abierto con realimentación unitaria sin controlador extra
# BC original: 4/(s^2 + 2s + 5 + 4) = 4/(s^2 + 2s + 9)
sys_orig = signal.TransferFunction([4], [1, 2, 9])
t1, y_orig = signal.step(sys_orig, T=t)

# Con PD
Kp_ex2 = 0.394
Kd_ex2 = 0.5
num_pd = [4 * Kd_ex2, 4 * Kp_ex2]
den_pd = [1, 2 + 4 * Kd_ex2, 5 + 4 * Kp_ex2]
sys_pd = signal.TransferFunction(num_pd, den_pd)
t2, y_pd = signal.step(sys_pd, T=t)

ax.plot(t1, y_orig, color='gray', lw=2, ls='--', label='BC sin controlador extra')
ax.plot(t2, y_pd, color=COLOR_PUNTO, lw=2.5, label=rf'Con PD ($K_p={Kp_ex2}$, $K_d={Kd_ex2}$)')
ax.axhline(1, color=COLOR_RECTA, ls=':', lw=1.5, alpha=0.5)
ax.axhspan(0.98, 1.02, alpha=0.1, color=COLOR_PUNTO, label=r'Banda $\pm 2\%$')
ax.set_xlabel('Tiempo (s)')
ax.set_ylabel(r'Salida $y(t)$')
ax.set_title(r'Ejercicio 2: Diseño PD para $G(s) = 4/(s^2+2s+5)$')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

### 7.3 Ejercicio 3: Análisis del lugar de las raíces

#### Ejercicio resuelto: Lugar de raíces de $G(s) = (s+3)/[s(s+5)]$

**Datos:** $G(s) = \dfrac{s + 3}{s(s + 5)}$. Trazar el lugar de las raíces y determinar el rango de $K$ para estabilidad.

**Paso 1:** Identificar polos y ceros de lazo abierto:
- Polos: $s = 0$, $s = -5$ (2 polos)
- Ceros: $s = -3$ (1 cero)
- $n - m = 2 - 1 = 1$ asíntota

**Paso 2:** Asíntota:
- Centro: $\sigma_a = \dfrac{(0 + (-5)) - (-3)}{2 - 1} = -2$
- Ángulo: $180°$

**Paso 3:** Tramos en el eje real: entre $s = 0$ y $s = -3$ (1 polo+cero a la derecha = impar) y $s < -5$ (3 polos+ceros a la derecha = impar).

**Paso 4:** Punto de separación. $K = -\dfrac{D(s)}{N(s)} = -\dfrac{s(s+5)}{s+3}$. Derivar e igualar a cero:

$$\frac{dK}{ds} = -\frac{(2s + 5)(s + 3) - s(s + 5)}{(s + 3)^2} = 0$$

$$s^2 + 6s + 15 = 0 \implies s = \frac{-6 \pm \sqrt{36 - 60}}{2}$$

Discriminante negativo $\to$ **no hay puntos de separación reales**. Las ramas van directamente del eje real al cero/infinito.

**Paso 5:** Estabilidad. Como todas las ramas permanecen en el semiplano izquierdo (una termina en $s = -3$ y la asíntota va a $-\infty$), el sistema es **estable para todo $K > 0$**.

In [ ]:
# Lugar de las raíces: G(s) = (s+3)/[s(s+5)]
fig, ax = plt.subplots(figsize=(10, 8))

num_coeffs = [1, 3]         # s + 3
den_coeffs = [1, 5, 0]     # s^2 + 5s = s(s+5)

K_values = np.concatenate([
    np.linspace(0.001, 5, 500),
    np.linspace(5, 50, 500),
    np.linspace(50, 500, 500)
])

all_roots = []
for K in K_values:
    num_padded = np.zeros(len(den_coeffs))
    num_padded[-len(num_coeffs):] = num_coeffs
    char_poly = np.array(den_coeffs) + K * num_padded
    roots = np.roots(char_poly)
    all_roots.append(roots)

all_roots = np.array(all_roots)

colores_ramas = [COLOR_PRINCIPAL, COLOR_RECTA]
for i in range(all_roots.shape[1]):
    ax.plot(all_roots[:, i].real, all_roots[:, i].imag, '.', color=colores_ramas[i % 2],
            ms=1.5, alpha=0.6)

# Polos y ceros de LA
polos_la = np.roots(den_coeffs)
ceros_la = np.roots(num_coeffs)
ax.plot(polos_la.real, polos_la.imag, 'x', color='k', ms=14, mew=3, zorder=5, label='Polos LA')
ax.plot(ceros_la.real, ceros_la.imag, 'o', color='k', ms=12, mew=3, fillstyle='none', zorder=5, label='Ceros LA')

ax.set_xlabel(r'$\mathrm{Re}(s)$')
ax.set_ylabel(r'$\mathrm{Im}(s)$')
ax.set_title(r'Lugar de las raíces: $G(s) = (s+3)/[s(s+5)]$')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-8, 2)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---

### 7.4 Ejercicio 4: Análisis de cancelación

#### Ejercicio resuelto: Cancelación en $G(s) = 1/[(s-2)(s+4)]$ con PI

**Datos:** $G(s) = \dfrac{1}{(s - 2)(s + 4)}$. Un ingeniero propone un controlador $C(s) = K \cdot \dfrac{s - 2}{s}$ para cancelar el polo inestable.

**Paso 1:** El polo $s = 2$ está en el **semiplano derecho** $\to$ polo **inestable**.

**Paso 2:** Tras la "cancelación":

$$C(s) G(s) = \frac{K \cancel{(s - 2)}}{s \cancel{(s - 2)} (s + 4)} = \frac{K}{s(s + 4)}$$

La FT de BC parece estable para $K$ apropiado.

**Paso 3:** Pero la realización interna conserva el modo $e^{2t}$. Ante la más mínima perturbación, la salida real diverge.

**Conclusión:**

$$\boxed{\text{Cancelación INVÁLIDA: el polo } s = 2 \text{ es inestable.}}$$

**Solución correcta:** usar un controlador que estabilice el polo inestable mediante realimentación, no mediante cancelación. Por ejemplo, un controlador PD con ganancia suficiente para desplazar todos los polos al semiplano izquierdo.

---

### 7.5 Ejercicio 5: Diseño PID completo

#### Ejercicio resuelto: PID para $G(s) = 10/[(s+1)(s+5)]$

**Datos:** $G(s) = \dfrac{10}{(s + 1)(s + 5)}$. Diseñar PID para $e_{ss} = 0$, $SO \leq 10\%$, $t_s \leq 2$ s.

**Paso 1:** Especificaciones:

$$\zeta = \frac{-\ln(0.10)}{\sqrt{\pi^2 + \ln^2(0.10)}} = 0.591$$

$$\sigma = \frac{4}{2} = 2 \implies \omega_n = \frac{2}{0.591} = 3.384$$

**Paso 2:** PID con cancelación de ambos polos:

$$C(s) = K_c \cdot \frac{(s + 1)(s + 5)}{s}$$

Tras cancelación:

$$G_{LA}(s) = \frac{10 K_c}{s}$$

$$G_{BC}(s) = \frac{10 K_c}{s + 10 K_c}$$

Esto es de 1er orden con polo en $s = -10 K_c$. **No hay sobreoscilación** (1er orden no oscila).

**Paso 3:** Imponer $t_s \leq 2$ s:

$$t_s = \frac{4}{10 K_c} \leq 2 \implies K_c \geq 0.2$$

**Paso 4:** Elegimos $K_c = 0.3$:

$$\boxed{C(s) = 0.3 \cdot \frac{(s+1)(s+5)}{s} = 0.3 \cdot \frac{s^2 + 6s + 5}{s}}$$

**Verificación:**
- $G_{BC}(s) = \dfrac{3}{s + 3}$ $\to$ polo en $s = -3$ $\to$ $t_s = \dfrac{4}{3} = 1.33$ s $\leq 2$ s $\checkmark$
- Sistema tipo 1 (integrador en el lazo) $\to$ $e_{ss} = 0$ $\checkmark$
- 1er orden $\to$ $SO = 0\%$ $\leq 10\%$ $\checkmark$

**Nota:** la cancelación es válida porque ambos polos ($s = -1$ y $s = -5$) son **estables**.

In [ ]:
# Verificación del ejercicio 5: PID completo
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

t = np.linspace(0, 5, 500)

# Sin controlador (realimentación unitaria)
num_orig = [10]
den_orig_cl = [1, 6, 5 + 10]  # s^2 + 6s + 15
sys_orig = signal.TransferFunction(num_orig, den_orig_cl)
t1, y_orig = signal.step(sys_orig, T=t)

# Con PID
Kc = 0.3
y_pid = 1 - np.exp(-3 * t)

axes[0].plot(t1, y_orig, color='gray', lw=2, ls='--', label='Sin controlador (solo realim.)')
axes[0].plot(t, y_pid, color=COLOR_PUNTO, lw=2.5, label=r'Con PID ($K_c = 0.3$)')
axes[0].axhline(1, color=COLOR_RECTA, ls=':', lw=1.5, alpha=0.5)
axes[0].axhspan(0.98, 1.02, alpha=0.1, color=COLOR_PUNTO)
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel(r'Salida $y(t)$')
axes[0].set_title(r'Respuesta PID: $G(s) = 10/[(s+1)(s+5)]$')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.3)

# Mapa de polos
axes[1].axhline(0, color='k', lw=0.8)
axes[1].axvline(0, color='k', lw=0.8)

# Polos originales del BC sin controlador
polos_orig = np.roots(den_orig_cl)
axes[1].plot(polos_orig.real, polos_orig.imag, 'x', color='gray', ms=14, mew=3, label='Polos BC original')

# Polo con PID
axes[1].plot(-3, 0, 'x', color=COLOR_PUNTO, ms=14, mew=3, label=r'Polo BC con PID: $s = -3$')

# Ceros del PID que cancelan
axes[1].plot(-1, 0, 'o', color=COLOR_CERO, ms=12, mew=2, fillstyle='none', label='Ceros PID (cancelan polos)')
axes[1].plot(-5, 0, 'o', color=COLOR_CERO, ms=12, mew=2, fillstyle='none')

axes[1].set_xlabel(r'$\mathrm{Re}(s)$')
axes[1].set_ylabel(r'$\mathrm{Im}(s)$')
axes[1].set_title('Mapa de polos y ceros: diseño PID')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-4, 4)

plt.tight_layout()
plt.show()

---

## 8. Lugar de las raíces: ejemplos adicionales

### 8.1 Lugar de raíces con cero: $G(s) = (s+2)/[s^2(s+4)]$

In [ ]:
# Lugar de raíces: G(s) = (s+2)/[s^2(s+4)]
fig, ax = plt.subplots(figsize=(10, 8))

num_coeffs = [1, 2]             # s + 2
den_coeffs = [1, 4, 0, 0]      # s^3 + 4s^2 = s^2(s+4)

K_values = np.concatenate([
    np.linspace(0.001, 2, 500),
    np.linspace(2, 20, 500),
    np.linspace(20, 200, 500)
])

all_roots = []
for K in K_values:
    num_padded = np.zeros(len(den_coeffs))
    num_padded[-len(num_coeffs):] = num_coeffs
    char_poly = np.array(den_coeffs) + K * num_padded
    roots = np.roots(char_poly)
    all_roots.append(roots)

all_roots = np.array(all_roots)

colores_ramas = [COLOR_PRINCIPAL, COLOR_RECTA, COLOR_PUNTO]
for i in range(all_roots.shape[1]):
    ax.plot(all_roots[:, i].real, all_roots[:, i].imag, '.', color=colores_ramas[i % 3],
            ms=1.5, alpha=0.6)

# Polos y ceros
polos_la = np.roots(den_coeffs)
ceros_la = np.roots(num_coeffs)
ax.plot(polos_la.real, polos_la.imag, 'x', color='k', ms=14, mew=3, zorder=5, label='Polos LA')
ax.plot(ceros_la.real, ceros_la.imag, 'o', color='k', ms=12, mew=3, fillstyle='none', zorder=5, label='Ceros LA')

# Asíntotas: n-m = 3-1 = 2
sigma_a = (0 + 0 + (-4) - (-2)) / 2
ax.axvline(sigma_a, color='gray', ls=':', lw=1, alpha=0.5,
           label=rf'Centro asíntotas: $\sigma_a = {sigma_a:.1f}$')

ax.set_xlabel(r'$\mathrm{Re}(s)$')
ax.set_ylabel(r'$\mathrm{Im}(s)$')
ax.set_title(r'Lugar de las raíces: $G(s) = (s+2)/[s^2(s+4)]$')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xlim(-8, 2)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### 8.2 Lugar de raíces con polos complejos: $G(s) = 1/(s^2 + 2s + 5)$

In [ ]:
# Lugar de raíces con polos complejos: G(s) = 1/(s^2 + 2s + 5)
fig, ax = plt.subplots(figsize=(10, 8))

num_coeffs = [1]
den_coeffs = [1, 2, 5]

K_values = np.concatenate([
    np.linspace(0.001, 5, 500),
    np.linspace(5, 50, 500),
    np.linspace(50, 500, 500)
])

all_roots = []
for K in K_values:
    num_padded = np.zeros(len(den_coeffs))
    num_padded[-len(num_coeffs):] = num_coeffs
    char_poly = np.array(den_coeffs) + K * num_padded
    roots = np.roots(char_poly)
    all_roots.append(roots)

all_roots = np.array(all_roots)

colores_ramas = [COLOR_PRINCIPAL, COLOR_RECTA]
for i in range(all_roots.shape[1]):
    ax.plot(all_roots[:, i].real, all_roots[:, i].imag, '.', color=colores_ramas[i % 2],
            ms=1.5, alpha=0.6)

# Polos de LA
polos_la = np.roots(den_coeffs)
ax.plot(polos_la.real, polos_la.imag, 'x', color='k', ms=14, mew=3, zorder=5, label='Polos LA')

for p in polos_la:
    ax.annotate(f'$s = {p.real:.0f}{p.imag:+.0f}j$', xy=(p.real, p.imag),
                xytext=(p.real + 1.5, p.imag + 0.5), fontsize=10, color='k',
                arrowprops=dict(arrowstyle='->', lw=1.5))

# Punto de ruptura en el eje real
# K = -(s^2 + 2s + 5), dK/ds = -(2s + 2) = 0 -> s = -1
# K(-1) = -(1 - 2 + 5) = -4 < 0 -> no está en el lugar (K > 0)
# Las ramas van verticalmente hacia las asíntotas a ±90°

ax.set_xlabel(r'$\mathrm{Re}(s)$')
ax.set_ylabel(r'$\mathrm{Im}(s)$')
ax.set_title(r'Lugar de las raíces: $G(s) = 1/(s^2 + 2s + 5)$')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5, 3)
ax.set_ylim(-8, 8)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

**Observación importante:** cuando los polos de LA son complejos conjugados y no hay ceros, las ramas se alejan verticalmente (asíntotas a $\pm 90°$). El sistema se vuelve menos amortiguado al aumentar $K$ pero **nunca se inestabiliza** (las ramas permanecen en el SPIzq con $\mathrm{Re}(s) = -1$ constante).

---

## 9. Catálogo completo de ejercicios: todos los patrones

### Tabla resumen de tipos

| # | Tipo de diseño | Planta | Controlador | Ecuación clave | Dificultad |
|---|----------------|--------|-------------|----------------|------------|
| 1 | P para 1er orden | $\dfrac{K}{\tau s + 1}$ | $K_p$ | $e_{ss} = \dfrac{1}{1 + KK_p}$ | Baja |
| 2 | PI para 1er orden (cancelación) | $\dfrac{K}{\tau s + 1}$ | $K_p \dfrac{\tau s + 1}{\tau s}$ | $\tau_{BC} = \dfrac{\tau}{KK_p}$ | Baja |
| 3 | PI para 1er orden (sin cancelar) | $\dfrac{K}{\tau s + 1}$ | $K_p \dfrac{T_I s + 1}{T_I s}$ | $s^2 + \ldots$ (2o orden) | Media |
| 4 | PD para 2o orden | $\dfrac{K}{s^2 + as + b}$ | $K_p + K_d s$ | Igualar coefs. con $D(s)$ | Media |
| 5 | PID cancelación total | $\dfrac{K}{(s+p_1)(s+p_2)}$ | $K_c \dfrac{(s+p_1)(s+p_2)}{s}$ | $\tau_{BC} = \dfrac{1}{KK_c}$ | Media |
| 6 | PID cancelación parcial | $\dfrac{K}{(s+p_1)(s+p_2)}$ | $K_c \dfrac{(s+p_1)(T_I s+1)}{T_I s}$ | Igualar coefs. | Alta |
| 7 | Lugar de raíces: encontrar $K_{crit}$ | Cualquiera | $K$ | Routh-Hurwitz | Media |
| 8 | Lugar de raíces: elegir $K$ para specs | Cualquiera | $K$ | Intersección LR con región | Media |
| 9 | Análisis de cancelación | Con polo inestable | Dado | Verificar SPIzq/SPDer | Baja |
| 10 | Efecto de ceros | Con cero adicional | Dado | Comparar con/sin cero | Baja |

---

### 9.1 Tipo 1: Control P para sistema de 1er orden

$$\boxed{G_{BC}(s) = \frac{KK_p}{\tau s + 1 + KK_p}} \qquad \boxed{e_{ss} = \frac{1}{1 + KK_p}}$$

**Cómo afectan los parámetros:**
- Si **$K_p$ aumenta** $\to$ $\tau_{BC}$ disminuye (más rápido) $\to$ $e_{ss}$ disminuye (pero nunca llega a 0)
- Si **$K_p \to \infty$** $\to$ $e_{ss} \to 0$ pero la señal de control $u(t)$ crece sin límite (saturación del actuador)

**Paso a paso:**
1. Identificar $K$ y $\tau$ de la planta
2. Calcular $K_p$ para cumplir $e_{ss} \leq e_{max}$: $K_p \geq \dfrac{1 - e_{max}}{K \cdot e_{max}}$
3. Verificar $t_s = \dfrac{4\tau}{1 + KK_p}$

**Limitación:** no puede lograr $e_{ss} = 0$ en sistema tipo 0.

---

### 9.2 Tipo 2: PI con cancelación para 1er orden

$$C(s) = K_p \cdot \frac{\tau s + 1}{\tau s} \qquad \boxed{G_{BC}(s) = \frac{KK_p/\tau}{s + KK_p/\tau}}$$

**Cómo afectan los parámetros:**
- Si **$K_p$ aumenta** $\to$ polo BC se aleja del origen $\to$ más rápido
- **$T_I = \tau$** es obligatorio para la cancelación

**Paso a paso:**
1. Identificar $K$ y $\tau$. Fijar $T_I = \tau$
2. Tras cancelación: polo BC en $s = -KK_p/\tau$
3. Imponer $t_s$: $K_p = \dfrac{4\tau}{K \cdot t_s}$
4. $e_{ss} = 0$ automáticamente (tipo 1)

**Ventaja:** diseño sencillo, 1 solo parámetro libre.

**Desventaja:** depende del conocimiento exacto de $\tau$.

---

### 9.3 Tipo 3: PI sin cancelación para 1er orden

$$C(s) = K_p \cdot \frac{T_I s + 1}{T_I s}$$

Sin cancelación, el BC es de **2o orden**:

$$G_{BC}(s) = \frac{KK_p(T_I s + 1)}{\tau T_I s^2 + T_I(1 + KK_p)s + KK_p}$$

$$\boxed{\omega_n = \sqrt{\frac{KK_p}{\tau T_I}}} \qquad \boxed{\zeta = \frac{T_I(1 + KK_p)}{2\sqrt{\tau T_I KK_p}}}$$

**Paso a paso:**
1. De las specs, obtener $\zeta_{des}$ y $\omega_{n,des}$
2. De $\omega_n$: $K_p = \dfrac{\tau T_I \omega_n^2}{K}$
3. De $\zeta$: $T_I = \dfrac{2\zeta\sqrt{\tau T_I KK_p}}{1 + KK_p}$ (resolver iterativamente o sustituir)
4. Verificar que el cero del PI no distorsione demasiado la respuesta

**Cuidado:** el cero en $s = -1/T_I$ aumenta la sobreoscilación respecto a un 2o orden puro.

---

### 9.4 Tipo 4: PD para sistema de 2o orden (asignación de polos)

$$C(s) = K_p + K_d s \qquad \text{EC: } s^2 + (a_1 + K K_d)s + (a_0 + KK_p) = 0$$

$$\boxed{K_d = \frac{2\zeta_{des}\omega_{n,des} - a_1}{K}} \qquad \boxed{K_p = \frac{\omega_{n,des}^2 - a_0}{K}}$$

**Cómo afectan los parámetros:**
- $K_d$ controla el **amortiguamiento** (coeficiente de $s$)
- $K_p$ controla la **frecuencia natural** (término independiente)

**Paso a paso:**
1. Obtener $\zeta_{des}$ y $\omega_{n,des}$ de las specs
2. Igualar coeficientes del polinomio característico
3. Resolver para $K_p$ y $K_d$
4. **Verificar** que $K_d > 0$ (realizable) y que el cero del PD no sea problemático

**Limitación:** no elimina el error estacionario (sistema sigue siendo tipo 0).

---

### 9.5 Tipo 5: PID con cancelación total para 2o orden

$$C(s) = K_c \cdot \frac{(s + p_1)(s + p_2)}{s} \qquad \boxed{G_{BC}(s) = \frac{KK_c}{s + KK_c}}$$

**Paso a paso:**
1. Factorizar la planta: $G(s) = K / [(s + p_1)(s + p_2)]$
2. Los ceros del PID cancelan los polos: $C(s) = K_c (s+p_1)(s+p_2)/s$
3. Resultado: 1er orden con $\tau_{BC} = 1/(KK_c)$
4. De $t_s$: $K_c = \dfrac{4}{K \cdot t_s}$

**Ventaja:** diseño muy simple, sin sobreoscilación.

**Requisito:** ambos polos deben ser **estables** ($p_1, p_2 > 0$).

---

### 9.6 Tipo 6: PID con cancelación parcial

Se cancela solo un polo (el más lento, por ejemplo $p_1$):

$$C(s) = K_c \cdot \frac{(s + p_1)(T_I s + 1)}{T_I s}$$

Tras cancelar $s + p_1$:

$$G_{LA}(s) = \frac{K K_c (T_I s + 1)}{T_I s (s + p_2)}$$

El BC es de **2o orden** con un cero.

**Paso a paso:**
1. Cancelar el polo lento con un cero del PID
2. Elegir $T_I$ y $K_c$ para cumplir $\zeta$ y $\omega_n$ deseados
3. Igualar coeficientes del polinomio de 2o orden resultante
4. Considerar el efecto del cero restante ($s = -1/T_I$)

---

### 9.7 Tipo 7: Determinar $K_{crit}$ con el lugar de las raíces

$$\boxed{K_{crit} = K \text{ tal que los polos del BC cruzan el eje imaginario}}$$

**Paso a paso (método de Routh):**
1. Escribir la ecuación característica: $D(s) + K N(s) = 0$
2. Construir la tabla de Routh
3. Encontrar $K$ que hace cero una fila completa
4. Los polos en el eje imaginario se obtienen del polinomio auxiliar

**Paso a paso (método gráfico):**
1. Trazar el lugar de las raíces
2. Identificar el cruce con el eje imaginario
3. Leer $K$ en ese punto

**Ejemplo clave:** para $G(s) = 1/[s(s+1)(s+2)]$, $K_{crit} = 6$ (determinado por Routh: fila de $s^1$ se anula con $K = 6$).

---

### 9.8 Tipo 8: Elegir $K$ del lugar de raíces para cumplir especificaciones

**Paso a paso:**
1. Trazar las regiones de especificaciones en el plano $s$ ($\sigma_{min}$, $\theta_{\zeta}$)
2. Trazar el lugar de las raíces
3. Encontrar la intersección del LR con la frontera de la región
4. Calcular $K$ en ese punto: $K = \dfrac{|D(s_0)|}{|N(s_0)|}$ donde $s_0$ es la intersección

**Truco:** si el LR cruza la línea $\zeta = \zeta_{des}$, sustituir $s = \omega_n(-\zeta + j\sqrt{1-\zeta^2})$ en la ecuación $1 + KG(s) = 0$ y resolver $K$ y $\omega_n$.

In [ ]:
# Tipo 8: elegir K en el lugar de raíces para especificaciones
fig, ax = plt.subplots(figsize=(10, 8))

# G(s) = 1/[s(s+4)]
num_coeffs = [1]
den_coeffs = [1, 4, 0]

K_values = np.concatenate([
    np.linspace(0.001, 4, 500),
    np.linspace(4, 20, 500),
    np.linspace(20, 100, 500)
])

all_roots = []
for K in K_values:
    num_padded = np.zeros(len(den_coeffs))
    num_padded[-len(num_coeffs):] = num_coeffs
    char_poly = np.array(den_coeffs) + K * num_padded
    roots = np.roots(char_poly)
    all_roots.append(roots)

all_roots = np.array(all_roots)

# Dibujar LR
for i in range(all_roots.shape[1]):
    ax.plot(all_roots[:, i].real, all_roots[:, i].imag, '.', color=COLOR_PRINCIPAL,
            ms=1.5, alpha=0.6)

# Región de especificaciones: zeta >= 0.7, ts <= 2s (sigma >= 2)
zeta_des = 0.7
sigma_des = 2.0
theta_des = np.arccos(zeta_des)

# Líneas de especificación
ax.axvline(-sigma_des, color=COLOR_RECTA, ls='--', lw=2, alpha=0.7, label=rf'$\sigma \geq {sigma_des}$')
r_line = 8
ax.plot([0, -r_line * np.cos(theta_des)], [0, r_line * np.sin(theta_des)],
        '--', color=COLOR_PUNTO, lw=2, alpha=0.7)
ax.plot([0, -r_line * np.cos(theta_des)], [0, -r_line * np.sin(theta_des)],
        '--', color=COLOR_PUNTO, lw=2, alpha=0.7, label=rf'$\zeta \geq {zeta_des}$')

# Encontrar K en la intersección con la línea de zeta
# s = wn(-zeta + j*sqrt(1-zeta^2))
# 1 + K/[s(s+4)] = 0 -> K = -s(s+4) = -s^2 - 4s
# s en la línea de zeta con sigma=2: s = -2 + j*2*tan(theta_des)
wd_intersect = sigma_des * np.tan(theta_des)
s_intersect = complex(-sigma_des, wd_intersect)
K_intersect = abs(-s_intersect * (s_intersect + 4))

ax.plot(s_intersect.real, s_intersect.imag, 'o', color=COLOR_PUNTO, ms=14, zorder=5)
ax.plot(s_intersect.real, -s_intersect.imag, 'o', color=COLOR_PUNTO, ms=14, zorder=5)
ax.annotate(f'$K = {K_intersect.real:.1f}$\n$s = {s_intersect.real:.1f}{s_intersect.imag:+.2f}j$',
            xy=(s_intersect.real, s_intersect.imag),
            xytext=(s_intersect.real + 1.5, s_intersect.imag + 1),
            fontsize=11, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2),
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))

# Polos LA
polos_la = np.roots(den_coeffs)
ax.plot(polos_la.real, polos_la.imag, 'x', color='k', ms=14, mew=3, zorder=5, label='Polos LA')

ax.set_xlabel(r'$\mathrm{Re}(s)$')
ax.set_ylabel(r'$\mathrm{Im}(s)$')
ax.set_title(r'Selección de $K$ en el LR: $G(s) = 1/[s(s+4)]$')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xlim(-8, 2)
ax.set_ylim(-6, 6)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---

### 9.9 Tipo 9: Análisis de cancelación polo-cero

**Paso a paso:**
1. Identificar si el polo que se quiere cancelar está en el **SPIzq** o **SPDer**
2. Si $\mathrm{Re}(polo) < 0$ $\to$ cancelación **aceptable** (modo decae)
3. Si $\mathrm{Re}(polo) \geq 0$ $\to$ cancelación **PROHIBIDA** (modo crece o no decae)
4. Incluso en cancelación aceptable: verificar que la cancelación sea razonablemente exacta. Si hay incertidumbre en el modelo, el modo "cancelado" puede persistir con amplitud no despreciable.

$$\boxed{\text{Regla: solo cancelar polos con } \mathrm{Re}(s) < 0}$$

---

### 9.10 Tipo 10: Efecto de ceros en la respuesta

**Paso a paso:**
1. Identificar el cero de la FT de BC: $s = -z$
2. Comparar $|z|$ con $|\sigma|$ de los polos dominantes
3. Si $|z| \approx |\sigma|$: el cero **afecta significativamente** la respuesta
   - Cero en SPIzq: **aumenta** sobreoscilación, **reduce** tiempo de subida
   - Cero en SPDer: produce **undershoot** (respuesta inversa inicial)
4. Si $|z| \gg |\sigma|$: efecto **despreciable**

**Fórmula aproximada** para el efecto de un cero en SPIzq sobre la sobreoscilación:

$$SO_{\text{con cero}} \approx SO_{\text{sin cero}} \cdot \left(1 + \frac{\sigma}{z}\right)$$

---

### Tabla resumen de fórmulas por tipo

| Tipo | Controlador | Ganancia BC | Polo(s) BC | Error $e_{ss}$ |
|------|------------|-------------|------------|----------------|
| 1. P + 1er orden | $K_p$ | $\dfrac{KK_p}{1+KK_p}$ | $s = -\dfrac{1+KK_p}{\tau}$ | $\dfrac{1}{1+KK_p}$ |
| 2. PI cancel. + 1er orden | $K_p\dfrac{\tau s+1}{\tau s}$ | $1$ | $s = -\dfrac{KK_p}{\tau}$ | $0$ |
| 4. PD + 2o orden | $K_p + K_d s$ | $< 1$ | Asignación directa | $\neq 0$ |
| 5. PID cancel. + 2o orden | $K_c\dfrac{(s+p_1)(s+p_2)}{s}$ | $1$ | $s = -KK_c$ | $0$ |

---

## 10. Resumen y tablas de referencia

### Conversión especificaciones $\to$ plano $s$

| Especificación | Fórmula |
|----------------|--------|
| $SO\% \to \zeta$ | $\zeta = \dfrac{-\ln(SO/100)}{\sqrt{\pi^2 + \ln^2(SO/100)}}$ |
| $t_s \to \sigma$ | $\sigma = \dfrac{4}{t_s}$ (criterio 2%) |
| $t_p \to \omega_d$ | $\omega_d = \dfrac{\pi}{t_p}$ |
| $\zeta, \sigma \to \omega_n$ | $\omega_n = \dfrac{\sigma}{\zeta}$ |
| Polos deseados | $s = -\sigma \pm j\omega_d$ |

### Lugar de las raíces: reglas rápidas

| Propiedad | Fórmula |
|-----------|--------|
| Inicio | Polos de $G(s)$ (para $K = 0$) |
| Final | Ceros de $G(s)$ o $\infty$ (para $K \to \infty$) |
| N. de ramas | $n$ = n. de polos |
| N. de asíntotas | $n - m$ |
| Centro asíntotas | $\sigma_a = \dfrac{\sum p_i - \sum z_j}{n - m}$ |
| Ángulos asíntotas | $\theta_k = \dfrac{(2k+1) \cdot 180°}{n - m}$, $k = 0, 1, \ldots$ |
| Eje real | Punto $s_0 \in$ LR si n. impar de polos+ceros a su derecha |
| Punto separación | Resolver $\dfrac{dK}{ds} = 0$ donde $K = -D(s)/N(s)$ |

### Cancelación polo-cero

| Situación | Resultado |
|-----------|----------|
| Cancelar polo estable | **Aceptable** (modo decae) |
| Cancelar polo inestable | **PROHIBIDO** (modo diverge) |
| Cancelar polo marginalmente estable ($j\omega$) | **Peligroso** (modo no decae) |

### Efecto de ceros

| Tipo de cero | Efecto en la respuesta |
|-------------|------------------------|
| SPIzq cercano | Aumenta $SO\%$, reduce $t_r$ |
| SPIzq lejano | Despreciable |
| SPDer (no mínima fase) | Produce *undershoot* |